
# APIs, the home game edition!

<a href="https://colab.research.google.com/github/go-fair-us/apireference/blob/master/code/NIAID_API_homeEdition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## About

This notebook is a companion to the [GoFAIR US API slides](https://docs.google.com/presentation/d/1DdbZIy5HzJu_gm3PCRmD6WPMXioywznUBgmXp1Be2c8/edit?slide=id.p#slide=id.p) developed for NIH/NIAID.


Test image link from github raw URL.

<img src="https://raw.githubusercontent.com/go-fair-us/apireference/refs/heads/master/docs/images/Gemini_Generated_Image_igm93igm93igm93i.png" width="450"  alt="Google Logo Smaller">


## From the Blueprint

For standardized machine access to metadata, APIs should expose metadata elements described in Table 1. In general, an API should meet these minimum objectives:

* **Metadata Encoding:** API responses should return metadata encoded in JSON-LD, at least as an option. This would follow the types and properties guidance in the minimal metadata specification above.
* **IRI (URL) Structure:** API endpoints should be designed as resource-oriented IRIs (e.g., /datasets/{dataset_id}), avoiding verbs and complex query parameters in the IRI structure. This ensures that the IRIs can function as persistent identifiers (IRIs) within the JSON-LD @id field, enabling seamless integration into knowledge graphs.
* **HTTP Method:** Metadata retrieval should be performed using the HTTP GET method.
* **Documentation:** API documentation should adhere to OpenAPI/Swagger specifications for machine-readability and ease of use.
An example of API-exposed metadata formatted to meet these objectives is given in Supplemental Table 7.


!pip install pyoxigraph
!pip install networkx
!pip install typing
!pip install requests-cache   # ← run this once
!pip install pyld
!pip install ipysigma



In [1]:
import sys

# Check if running in Colab
if 'google.colab' in sys.modules:
    print("Running in Google Colab - installing packages...")
    !pip install pyoxigraph -q
    !pip install networkx -q
    !pip install requests-cache -q
    !pip install pyld -q
    !pip install ipysigma -q
    !pip install playwright -q
    !pip install nest_asyncio -q
    !playwright install -q
    !playwright install chromium -q
    !playwright install-deps chromium -q
else:
    print("Not in Colab - skipping installs.")

Not in Colab - skipping installs.


In [2]:
import pandas as pd
import networkx as nx
import requests
import pyoxigraph
from typing import Optional, List
import requests_cache
import time
from ipysigma import Sigma
from pyld import jsonld
from pyoxigraph import RdfFormat
import json
import nest_asyncio

nest_asyncio.apply()  # allows nested event loops in Colab

# Requests and Responses

## The simple Request - Response Sequence

 👉 TRY THIS: change the end of the url variable from 1 to 1a to see the error code path

In [3]:
# 1. Define the address (The Endpoint)
url = "https://jsonplaceholder.typicode.com/todos/1"

# 2. Send the "GET" request
response = requests.get(url)

# 3. Check if it worked (Status Code 200 means Success)
if response.status_code == 200:
    data = response.json()  # 4. Convert the raw text into a Python Dictionary (JSON)
    print(json.dumps(data, indent=2))  # 5. Pretty print the response
else:
    print(f"Error: {response.status_code}")


{
  "userId": 1,
  "id": 1,
  "title": "delectus aut autem",
  "completed": false
}


## Here is an example with ImmPort API

In [4]:
def search_seronet_studies(age_range: str = "5-50") -> dict:
    base_url = "https://immport.org/data/query/api/search/seronet/study"
    headers = {"accept": "application/json"}
    params = {"ageRange": age_range}

    response = requests.get(base_url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()


age_range = "5-50"
print(f"Querying ImmPort SeroNet studies with ageRange={age_range}...\n")

data = search_seronet_studies(age_range)
print(json.dumps(data, indent=2))

Querying ImmPort SeroNet studies with ageRange=5-50...

{
  "took": 46,
  "timed_out": false,
  "_shards": {
    "total": 5,
    "successful": 5,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 258,
      "relation": "eq"
    },
    "max_score": 0.0,
    "hits": [
      {
        "_index": "seronet_dr61",
        "_id": "SDY2519",
        "_score": 0.0,
        "_source": {
          "study_name": "Temporal variations in the severity of COVID-19 illness by race and ethnicity",
          "research_focus": "Epidemiology",
          "study_identifier": "PMID34308124_study-01",
          "enrollment_start_date": "2020-03-04",
          "study_description": "Introduction: Early reports highlighted racial/ethnic disparities in the severity of COVID-19 seen across the USA; the extent to which these disparities have persisted over time remains unclear. Our research objective was to understand temporal trends in racial/ethnic variation in severity of COVID-19 il

## Let's load this into Pandas.

In [5]:
hits = data["hits"]["hits"]

# Extract just the _source dictionaries
records = [hit["_source"] for hit in hits if "_source" in hit]

# Create DataFrame
df = pd.DataFrame(records)

print(df.shape)          # e.g. (10, ~35) in your sample
# print(df.columns.tolist())  # Optional display column list
df.head(10)

(10, 19)


,study_name,research_focus,study_identifier,enrollment_start_date,study_description,clinical_study_design,number_of_study_subjects,pubmed_id,reported_health_condition,study_accession,sars_cov_2_vaccine_type,sars_cov_2_variant,minimum_age,publication_title,enrollment_end_date,maximum_age,genus_and_species,doi,experiment
0,Temporal variations in the severity of COVID-1...,Epidemiology,PMID34308124_study-01,2020-03-04,Introduction: Early reports highlighted racial...,Retrospective Cohort,1584,34308124,[COVID-19],SDY2519,[Not Applicable],[Not Applicable],42,Temporal variations in the severity of COVID-1...,2020-12-05,81,[Homo sapiens],10.21430/M3U0J3FOKP,NaN
1,SARS-CoV-2 -specific immune responses in boost...,Immune Response,PMID35389888_study-01,,BACKGROUND. Breakthrough SARS-CoV-2 infections...,Not Applicable,76,35389888,[COVID-19],SDY2239,"[Johnson & Johnson SARS-CoV-2 vaccine, Moderna...","[SARS-CoV-2 Delta; B.1.617.2, SARS-CoV-2 Omicr...",21,SARS-CoV-2 -specific immune responses in boost...,,62,[Homo sapiens],10.21430/M3Q3C0UDD8,[{'biospecimen_collection_point': ['Post-vacci...
2,COVID-19 vaccine booster dose needed to achiev...,Vaccine Response,PMID35605428_study-01,2020-12-01,Methods: We longitudinally enrolled 85 NH resi...,Longitudinal Study,133,35605428,[COVID-19],SDY2546,[Pfizer-BioNTech SARS-CoV-2 vaccine],"[SARS-CoV-2 Wuhan/2020, SARS-CoV-2 Omicron; B....",30,COVID-19 vaccine booster dose needed to achiev...,2021-01-31,99,[Homo sapiens],10.21430/m3r5h37fu9,[{'biospecimen_collection_point': ['Pre-vaccin...
3,Omicron variant Spike-specific antibody bindin...,Immune Response,PMID35289637_study-01,,The Omicron variant of SARS-CoV-2 has been sho...,Prospective Cohort,38,35289637,[COVID-19],SDY2042,"[Pfizer-BioNTech SARS-CoV-2 vaccine, Moderna S...","[SARS-CoV-2 Alpha; B.1.1.7, SARS-CoV-2 Delta; ...",20,Omicron variant Spike-specific antibody bindin...,,53,[Homo sapiens],10.21430/M3LUE504P0,[{'biospecimen_collection_point': ['Post-vacci...
4,Vaccines elicit highly conserved cellular immu...,Vaccine Response,PMID35102312_study-01,,The highly mutated SARS-CoV-2 Omicron (B.1.1.5...,Prospective Cohort,47,35102312,[COVID-19],SDY2559,"[Pfizer-BioNTech SARS-CoV-2 vaccine, Johnson &...","[SARS-CoV-2 Omicron; B.1.1.529, SARS-CoV-2 Wuh...",22,Vaccines elicit highly conserved cellular immu...,,67,[Homo sapiens],10.21430/m32bp19s8x,[{'biospecimen_collection_point': ['Post-vacci...
5,"Early non-neutralizing, afucosylated antibody ...",Immune Response,PMID35040666_study-01,,A damaging inflammatory response is implicated...,Prospective Cohort,234,35040666,[COVID-19],SDY2038,[Pfizer-BioNTech SARS-CoV-2 vaccine],[Not Applicable],18,"Early non-neutralizing, afucosylated antibody ...",,98,"[Mus musculus, Homo sapiens]",10.21430/M3V1ZYUVBN,[{'biospecimen_collection_point': ['Post-sympt...
6,BA.5 bivalent booster vaccination enhances neu...,Vaccine Response,PMID37990024_study-01,,This study reports that most patients with NSC...,Prospective Cohort,46,37990024,"[COVID-19, Lung Cancer]",SDY3001,"[Pfizer-BioNTech SARS-CoV-2 bivalent vaccine, ...","[SARS-CoV-2 WA1/2020, SARS-CoV-2 Omicron varia...",34,BA.5 bivalent booster vaccination enhances neu...,,79,[Homo sapiens],10.21430/M318VAYLD4,[{'biospecimen_collection_point': ['Post-vacci...
7,Vaccine protection against the SARS-CoV-2 Omic...,Vaccine Response,PMID35427477_study-01,,The rapid spread of the SARS-CoV-2 Omicron (B....,Not Applicable,30,35427477,[COVID-19],SDY2033,"[Johnson & Johnson SARS-CoV-2 vaccine, Pfizer-...","[SARS-CoV-2 Delta; B.1.617.2, SARS-CoV-2 Omicr...",3,Vaccine protection against the SARS-CoV-2 Omic...,,10,[Macaca fascicularis],10.21430/M36QZ9VXR6,"[{'biospecimen_collection_point': ['Naïve'], '..."
8,Estimated preventable COVID-19-associated deat...,Epidemiology,PMID37093505_study-01,2021-05-30,While some studies have previously estimated l...,Retrospective Cohort,,37093505,[COVID-19],SDY2482,[Not Applicable],[Not Applicable],18,Estimated preventable COVID-19-associated deat.


<img src="https://raw.githubusercontent.com/go-fair-us/apireference/refs/heads/master/docs/images/Gemini_Generated_Image_e0yaxke0yaxke0ya.png" width="700"  alt="Google Logo Smaller">

### Exploring the Properties of The Request–Response


# Web Architecture

The W3C best practices for data on the web provides approaches for exposing metadata via web pages.  Often this takes the form of putting JSON-LD into a web page via tags in the head elements of the document object model.

In [6]:
import json
import threading
from playwright.sync_api import sync_playwright

def get_json_ld(url: str) -> list:
    results = []

    def run():
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()
            page.goto(url, wait_until="networkidle")

            json_ld_docs = page.evaluate("""() => {
                const scripts = document.querySelectorAll('script[type="application/ld+json"]');
                return Array.from(scripts).map(s => {
                    try { return JSON.parse(s.textContent); }
                    catch { return null; }
                }).filter(Boolean);
            }""")

            browser.close()
            results.extend(json_ld_docs)

    thread = threading.Thread(target=run)
    thread.start()
    thread.join()
    return results

url = "https://www.dev.immport.org/shared/study/SDY2176/summary"
results = get_json_ld(url)
print(json.dumps(results, indent=2))

[
  {
    "@context": "https://schema.org/",
    "@type": "Dataset",
    "_id": "sdy2176",
    "@id": "https://www.immport.org/shared/study/SDY2176",
    "identifier": [
      {
        "@type": "PropertyValue",
        "propertyID": "ImmPort Study Accession",
        "value": "SDY2176"
      },
      {
        "@type": "PropertyValue",
        "propertyID": "DOI",
        "value": "10.21430/M39VJJKEMO"
      }
    ],
    "abstract": "The current pandemic of COVID-19 caused by severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2) highlights an urgent need to develop a safe, efficacious, and durable vaccine. Using a measles virus (rMeV) vaccine strain as the backbone, we developed a series of recombinant attenuated vaccine candidates expressing various forms of the SARS-CoV-2 spike (S) protein and its receptor binding domain (RBD) and evaluated their efficacy in cotton rat, IFNAR-/-mice, IFNAR-/--hCD46 mice, and golden Syrian hamsters. We found that rMeV expressing stabilized pre

# 🎯 Demonstrating Cool URIs & Good API Design
## Why this matters for NIAID/FAIR data
- [Cool URIs](https://www.w3.org/Provider/Style/URI) (Tim Berners-Lee) = stable, hackable, human-readable, persistent
- Directly supports FAIR Findability, Accessibility & Reusability
- Avoids the caching problems we saw with `?version=2`

```
=== Not So Cool ===
https://example.org/resources?type=dataset&version=2&filter=active

=== Cool & Hackable ===
https://example.org/v2/datasets/active

Anyone can guess these URLs:
→ /v2/datasets          (all datasets)
→ /v2/datasets/active   (just active ones)
→ /v2/datasets/123      (specific dataset)
```

#### Path vs parameter URL patterns

In [7]:
base = "https://jsonplaceholder.typicode.com"  # public demo API

print("=== Query-parameter versioning (breaks caching) ===")
r1 = requests.get(f"{base}/posts?version=2")
print(f"URL: {r1.url}")

print("\n=== Path versioning (clean & cache-friendly) ===")
r2 = requests.get(f"{base}/v2/posts")  # imagine this pattern
print(f"URL: {r2.url}")
print(f"request: {r2.request}")

print(f"response: {r2.content}")


=== Query-parameter versioning (breaks caching) ===
URL: https://jsonplaceholder.typicode.com/posts?version=2

=== Path versioning (clean & cache-friendly) ===
URL: https://jsonplaceholder.typicode.com/v2/posts
request: <PreparedRequest [GET]>
response: b'{}'


#### Request Response examples


In [8]:
# REQUEST: Print a full raw HTTP request string
request_str = f"{r2.request.method} {r2.request.url} HTTP/1.1\r\n"
for key, value in r2.request.headers.items():
    request_str += f"{key}: {value}\r\n"
if r2.request.body:
    request_str += "\r\n" + (r2.request.body.decode('utf-8') if isinstance(r2.request.body, bytes) else r2.request.body)
else:
    request_str += "\r\n"
print(request_str)

GET https://jsonplaceholder.typicode.com/v2/posts HTTP/1.1
User-Agent: python-requests/2.32.4
Accept-Encoding: gzip, deflate
Accept: */*
Connection: keep-alive




In [9]:
# RESPONSE: Print a full raw HTTP response string
response_str = f"HTTP/1.1 {r2.status_code} {r2.reason}\r\n"
for key, value in r2.headers.items():
    response_str += f"{key}: {value}\r\n"
response_str += "\r\n" + r2.text
print(response_str)

HTTP/1.1 404 Not Found
Date: Thu, 19 Mar 2026 22:00:36 GMT
Content-Type: application/json; charset=utf-8
Content-Length: 2
Connection: keep-alive
access-control-allow-credentials: true
Cache-Control: max-age=43200
etag: W/"2-vyGp6PvFo4RvsFtPoIWeCReyIC8"
expires: -1
nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}
pragma: no-cache
report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=clwQNVVASAYunhOATMoMiJPBeKabwpSsyDueCom5gLY%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1773957636"}],"max_age":3600}
reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=clwQNVVASAYunhOATMoMiJPBeKabwpSsyDueCom5gLY%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1773957636"
Server: cloudflare
vary: Origin, Accept-Encoding
via: 2.0 heroku-router
x-content-type-options: nosniff
x-powered-by: Express
x-ratelimit-limit: 1000
x-ratelimit-remaining: 998
x-ratelimit-reset: 1773957682
cf

## Caching Demo

In [10]:
# pip install requests-cache   # ← run this once

# Install cache (in-memory for demo)
requests_cache.install_cache(
    'demo_cache',
    backend='memory',
    expire_after=60,
    urls_expire_after={
        "*/posts?*version=*": requests_cache.DO_NOT_CACHE,  # ← never cache query-param versioned URLs
    }
)

# The above "urls_expire_after" tag may seem like a code hack to prove a point that
# might not always be true.  It's really just trying to illustrate that caching may be something
# you don't control over the full route a request might take.

print("=== CACHING DEMO: Query-param vs Path versioning ===\n")

def time_request(url, label):
    start = time.time()
    r = requests.get(url)
    duration = (time.time() - start) * 1000
    status = "✅ CACHE HIT" if r.from_cache else "❌ CACHE MISS"
    print(f"{label:25} | {status:15} | {duration:6.1f} ms | {r.url}")
    return r

# BAD pattern (same content, different query string → but still caches!)
# _ = to suppress return from time_request, or just don't return anything in the function  ;)
print("BAD: query-param versioning")
_ = time_request("https://jsonplaceholder.typicode.com/posts?version=1", "Call 1")
_ = time_request("https://jsonplaceholder.typicode.com/posts?version=1", "Call 2")
_ = time_request("https://jsonplaceholder.typicode.com/posts?version=2", "Call 3")

print("\nGOOD: path versioning (using /posts as example)")
_ = time_request("https://jsonplaceholder.typicode.com/posts", "Call 1")
_ = time_request("https://jsonplaceholder.typicode.com/posts", "Call 2")



=== CACHING DEMO: Query-param vs Path versioning ===

BAD: query-param versioning
Call 1                    | ❌ CACHE MISS    |  326.7 ms | https://jsonplaceholder.typicode.com/posts?version=1
Call 2                    | ❌ CACHE MISS    |   45.4 ms | https://jsonplaceholder.typicode.com/posts?version=1
Call 3                    | ❌ CACHE MISS    |   41.9 ms | https://jsonplaceholder.typicode.com/posts?version=2

GOOD: path versioning (using /posts as example)
Call 1                    | ❌ CACHE MISS    | 1215.6 ms | https://jsonplaceholder.typicode.com/posts
Call 2                    | ✅ CACHE HIT     |    1.6 ms | https://jsonplaceholder.typicode.com/posts


## Formats

Immport JSON-LD

In [11]:
def search_studies(
    term: str = "influenza vaccine",
    from_record: int = 0,
    page_size: int = 200,
    pre_tag: str = "<em>",
    post_tag: str = "</em>",
    sort_field_direction: str = "asc",
    condition_or_disease: Optional[List[str]] = None,
) -> dict:

    if condition_or_disease is None:
        condition_or_disease = ["asthma", "COVID-19"]

    url = "https://immport.org/data/query/api/search/study"
    headers = {"accept": "application/json"}
    params = {
        "term": term,
        "fromRecord": from_record,
        "pageSize": page_size,
        "preTag": pre_tag,
        "postTag": post_tag,
        "format": "json",
        "sortFieldDirection": sort_field_direction,
        "conditionOrDisease": ",".join(condition_or_disease),
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()

In [12]:
def to_dataframe(data: dict) -> pd.DataFrame:
    """Flatten the hits[].`_source` records into a DataFrame."""
    hits = data.get("hits", {}).get("hits", [])
    records = [hit["_source"] for hit in hits if "_source" in hit]
    return pd.DataFrame(records)

data = search_studies()

df = to_dataframe(data)

In [13]:
df

,clinical_trial,gender,ethnicity,brief_title,initial_data_release_version_number,program_name,latest_data_release_version,arm_name,lab_test_panel_count,assay_method_count,...,study_pi,species,initial_data_release_version,condition_or_disease,has_assessment,assessment_panel_count,age_unit,assay_method,min_age,doi
0,N,[],[],COVID-19 vaccination enhances the immunogenici...,57.0,[CIVICs Collaborative Influenza Vaccine Innova...,DR58,"[2022-2023 Cohort, 2023-2024 Cohort, UGA8 2023...",[],[],...,"[Engin Berber: Lerner Research Institute, Ted ...",[],DR57,"[influenza, COVID-19]",N,[],Years,[],18.00,10.21430/M37OAP2WQW
1,N,[],[],Co-administration of seasonal quadrivalent inf...,54.1,[CIVICs Collaborative Influenza Vaccine Innova...,DR59,[Prime only. Seasonal inactivated influenza QI...,[],[],...,[Florian Krammer: Icahn School of Medicine at ...,[],DR54.1,"[influenza, COVID-19]",N,[],Not Specified,[],NaN,10.21430/M3DYPQ4GBE
2,N,[Unknown],[Not Specified],Concomitant administration of seasonal influen...,53.0,[CIVICs Collaborative Influenza Vaccine Innova...,DR58,"[Flu and COVID vaccines in same arm, Flu and C...",[],"[ELISA (1131), Hemagglutination Inhibition (15...",...,[Adolfo Garcia-Sastre: Icahn School of Medicin...,[Homo sapiens],DR53,[COVID-19],N,[],Years,"[Hemagglutination Inhibition, ELISA]",21.00,10.21430/M3N0YZN1L0
3,N,[],[],Concurrent Administration of COVID-19 and Infl...,53.1,[SeroNet],DR58,[Individuals who received influenza vaccine an...,[],"[Cell Mediated Immunoassay (0), Multiplex Immu...",...,"[Ryan Mcnamara: Ragon Institute of Mgh, Mit, A...",[],DR53.1,[COVID-19],N,[],Years,"[Pseudovirus Neutralization Assay, Cell Mediat...",23.00,10.21430/M3ECEGE6PK
4,N,[],[],A Virion-Based Combination Vaccine Protects ag...,53.0,[CIVICs Collaborative Influenza Vaccine Innova...,DR58,[Inactivated TM-RBD-HA prime boost and SARS2 c...,[],[],...,"[Nicholas Heaton: Duke University, Brook Heato...",[],DR53,"[influenza, COVID-19]",N,[],Weeks,[],NaN,10.21430/M3Y1UZJBP5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,N,[],[],Estimated preventable COVID-19-associated deat...,50.1,[SeroNet],DR58,[Preventable COVID-19 associated deaths],[],[],...,[Bill Hanage: Harvard T.h. Chan School of Publ...,[],DR50.1,[COVID-19],N,[],Years,[],18.00,10.21430/M3MTSYRBG6
196,N,[],[],Neutralization escape by SARS-CoV-2 Omicron su...,50.2,[SeroNet],DR58,"[No Bivalent mRNA Boost., Bivalent mRNA Boost.]",[],[Pseudovirus Neutralization Assay (0)],...,[Dan Barouch: Beth Israel Deaconess Medical Ce...,[],DR50.2,[COVID-19],N,[],Years,[Pseudovirus Neutralization Assay],23.00,10.21430/M38YJ0KH68
197,N,[],[],Substantial Neutralization Escape by SARS-CoV-...,51.2,[SeroNet],DR58,[Participantsboosted with monovalent mRNA boos...,[],[Pseudovirus Neutralization Assay (0)],...,[Dan Barouch: Beth Israel Deaconess Medical Ce...,[],DR51.2,[COVID-19],N,[],Years,[Pseudovirus Neutralization Assay],23.00,10.21430/M35KFIBVLJ
198,N,[Not Specified],[None],D614G Spike Mutation Increases SARS CoV-2 Susc...,48.0,[CIVICs Collaborative Influenza Vaccine Innova...,DR58,"[30ug intramuscular group, 30ug intradermal gr...",[],[Virus Neutralization (408)],...,"[Drew Weissman: University of Pennsylvania, Da...","[Macaca mulatta, Mus musculus]",DR48,[COVID-19],N,[],Weeks,[Virus Neutralization],0.15,10.21430/M32KSB7VCX


In [14]:
def fetch_jsonld(accession):
    url = f"https://s3.immport.org/release/metadata/Study_{accession}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            return r.json()  # ← text, not .json()
        return None
    except Exception:
        return None


df["jsonld"] = df["study_accession"].apply(fetch_jsonld)
df[["study_accession", "jsonld"]]


,study_accession,jsonld
0,SDY3233,"{'@context': 'https://schema.org/', '@type': '..."
1,SDY2932,"{'@context': 'https://schema.org/', '@type': '..."
2,SDY2752,"{'@context': 'https://schema.org/', '@type': '..."
3,SDY2845,"{'@context': 'https://schema.org/', '@type': '..."
4,SDY2746,"{'@context': 'https://schema.org/', '@type': '..."
...,...,...
195,SDY2482,"{'@context': 'https://schema.org/', '@type': '..."
196,SDY2520,"{'@context': 'https://schema.org/', '@type': '..."
197,SDY2639,"{'@context': 'https://schema.org/', '@type': '..."
198,SDY2238,"{'@context': 'https://schema.org/', '@type': '..."


In [15]:
def load_jsonld_to_oxigraph(store, df, column: str):
    """
    Load JSON-LD strings from a DataFrame column into an Oxigraph store.
    Expands remote @context references via pyld before loading.
    """

    for _, row in df.iterrows():
        raw = row[column]
        if not raw:
            continue

        if isinstance(raw, dict):
            doc = raw  # already a dict, no need to serialize/deserialize
        else:
            doc = json.loads(raw)

        doc['@context'] = "https://schema.org/docs/jsonldcontext.json"


        # Expand resolves all remote @context URLs, then re-serialize to N-Quads
        # expanded = jsonld.expand(doc)
        nquads = jsonld.to_rdf(doc, {"format": "application/n-quads"})

        store.load(nquads, RdfFormat.TURTLE, base_iri=None, to_graph=None)



In [16]:
mem_store = pyoxigraph.Store()

load_jsonld_to_oxigraph(mem_store, df, "jsonld")

In [17]:
# SPARQL
rq = """
PREFIX schema: <http://schema.org/>

SELECT  ?creator_name ?kw ?funder_name
 WHERE {
     ?s a schema:Dataset .
     ?s schema:creator ?creator .
     ?creator a schema:Person .
     ?creator schema:name ?creator_name .
     ?s schema:keywords ?kw .
     ?s schema:funding ?funding .
     ?funding schema:funder ?funder .
     ?funder a schema:Organization .
     ?funder schema:name ?funder_name
 }
"""


In [18]:
# This map is necessary to get the values from the pyoxigraph.QuerySolutions
def extract_value(cell):
    if isinstance(cell, (pyoxigraph.Literal, pyoxigraph.NamedNode, pyoxigraph.BlankNode)):
        return cell.value
    return cell

r = mem_store.query(rq)
results = list(r)

vars = r.variables
value_list = [variable.value for variable in vars]

rq_df = pd.DataFrame(results, columns=value_list)
rq_df = rq_df.map(extract_value)


In [19]:
rq_df

,creator_name,kw,funder_name
0,David Montefiori,SARS-CoV-2 Pseudovirus Neutralization Assay,National Institute of Allergy & Infectious Dis...
1,David Montefiori,SARS-CoV-2 Pseudovirus Neutralization Assay,Duke CIVIC Vaccine Center (DCVC)
2,David Montefiori,COVID-19,National Institute of Allergy & Infectious Dis...
3,David Montefiori,COVID-19,Duke CIVIC Vaccine Center (DCVC)
4,David Montefiori,CIVICs Collaborative Influenza Vaccine Innovat...,National Institute of Allergy & Infectious Dis...
...,...,...,...
677,Ted Ross,"Influenza HAI titers (seroconversion, seroposi...",Center for Influenza Vaccine Research for High...
678,Ted Ross,COVID-19,National Institute of Allergy & Infectious Dis...
679,Ted Ross,COVID-19,Center for Influenza Vaccine Research for High...
680,Ted Ross,CIVICs Collaborative Influenza Vaccine Innovat...,National Institute of Allergy & Infectious Dis...


In [20]:


# df = pd.read_csv("your_file.csv")  # or however you have your dataframe

G = nx.DiGraph()

for _, row in rq_df.iterrows():
    creator = str(row["creator_name"]).strip() if pd.notna(row["creator_name"]) else None
    kw = str(row["kw"]).strip() if pd.notna(row["kw"]) else None
    funder = str(row["funder_name"]).strip() if pd.notna(row["funder_name"]) else None

    if not creator:
        continue

    # Add nodes with type (so we can color them differently)
    G.add_node(creator, node_type="creator", label=creator)

    if kw:
        G.add_node(kw, node_type="keyword", label=kw)
        G.add_edge(creator, kw, relation="connects_to")

    if funder:
        G.add_node(funder, node_type="funder", label=funder)
        G.add_edge(creator, funder, relation="funded_by")

# Optional: add degree for sizing
degrees = dict(G.degree())
for node in G.nodes():
    G.nodes[node]["degree"] = degrees[node] + 3  # +3 for minimum visibility

print(f"Built graph with {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges")

Built graph with 152 nodes and 336 edges


In [24]:
# For Google Colab
if 'google.colab' in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()
else:
    print("Not in Colab - skipping installs.")

Not in Colab - skipping installs.


In [25]:

Sigma(
    G,
    node_color="node_type",           # colors creators blue, keywords orange, funders green (auto palette)
    node_size="degree",               # bigger = more connections
    edge_color="relation",            # different colors for "connects_to" vs "funded_by"
    node_label="label",
    height=800,
    start_layout=True                 # runs force-directed layout automatically
)

Sigma(nx.DiGraph with 152 nodes and 336 edges)